## Random Forest Classifier
Load features → check for leakage → weight features → train RF → evaluate.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

TRAIN_FEAT = "train_galaxy_features_improved.csv"
TRAIN_LBLS = "trainlabel.csv"
TEST_FEAT  = "test_galaxy_features_improved.csv"
TEST_LBLS  = "testlabel.csv"
CV_FOLDS   = 7
SEED       = 42


In [ ]:
def load_data(feat_csv, lbl_csv, name="Dataset"):
    print(f"\nLoading {name}")
    feats = pd.read_csv(feat_csv)
    lbls  = pd.read_csv(lbl_csv)
    lbls  = lbls.dropna(subset=['Label'])
    lbls['Label'] = lbls['Label'].astype(int)

    def name_from_file(fname):
        return fname.replace('norm_resized_','').replace('.fits','').replace('_',' ')
    feats['Target Name'] = feats['filename'].apply(name_from_file)

    merged = feats.merge(lbls, on='Target Name', how='inner')
    if len(merged) == 0:
        raise ValueError(f"No matched galaxies in {name}!")
    print(f"  {len(merged)} galaxies matched")

    feat_cols = [c for c in feats.columns if c not in ('filename','Target Name')]
    X = merged[feat_cols].values.astype(float)
    y = merged['Label'].values
    names = merged['Target Name'].values

    # fill NaN
    if np.isnan(X).any():
        cm = np.nanmean(X, axis=0)
        X[np.isnan(X)] = np.take(cm, np.where(np.isnan(X))[1])

    u, c = np.unique(y, return_counts=True)
    print(f"  Unbarred: {c[0]}  Barred: {c[1]}  Features: {len(feat_cols)}")
    return X, y, names, feat_cols

X_train, y_train, train_names, feat_names = load_data(TRAIN_FEAT, TRAIN_LBLS, "Training")
X_test,  y_test,  test_names,  _          = load_data(TEST_FEAT,  TEST_LBLS,  "Test")

# leakage check
overlap = set(train_names) & set(test_names)
if overlap:
    raise ValueError(f"DATA LEAKAGE: {len(overlap)} galaxies overlap!")
print(f"\nLeakage check passed ({len(train_names)} train, {len(test_names)} test)")


In [ ]:
# Feature weighting
weights = {
    'Nuclear_Concentration_Ratio': 2.5, 'Spiral_Symmetry_Score': 2.4,
    'Bar_Ansae_Test': 2.3,              'Edge_On_Indicator': 2.2,
    'Bar_Bulge_Ratio_Focused': 2.0,     'Bar_Fourier_Strength_Focused': 1.9,
    'Central_Asymmetry_180': 1.7,       'Central_Asymmetry_90': 1.6,
    'Concentration': 1.8, 'R50': 1.7, 'R90': 1.9,
    'Gini': 1.7, 'Ellipticity': 1.8, 'Radial_Profile_Ratio': 1.7,
}

Xtr_w = X_train.copy(); Xte_w = X_test.copy()
for i, feat in enumerate(feat_names):
    w = weights.get(feat, 1.0)
    Xtr_w[:, i] *= w; Xte_w[:, i] *= w

scaler = StandardScaler()
Xtr_s  = scaler.fit_transform(Xtr_w)
Xte_s  = scaler.transform(Xte_w)
print("Features weighted and scaled.")


In [ ]:
# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=1200, max_depth=18, max_features=0.65,
    min_samples_split=2, min_samples_leaf=1,
    bootstrap=True, class_weight='balanced',
    random_state=SEED, n_jobs=-1, max_samples=0.85
)
rf.fit(Xtr_s, y_train)
print("RF trained.")

# Optimal threshold search
proba_test = rf.predict_proba(Xte_s)[:, 1]
best_thr, best_tn, best_acc = 0.5, 0.0, 0.0
for thr in np.arange(0.45, 0.80, 0.01):
    pred_tmp = (proba_test >= thr).astype(int)
    cm_tmp   = confusion_matrix(y_test, pred_tmp)
    tn_rate  = cm_tmp[0,0]/(cm_tmp[0,0]+cm_tmp[0,1]+1e-6)
    acc_tmp  = (cm_tmp[0,0]+cm_tmp[1,1])/cm_tmp.sum()
    if tn_rate > best_tn and acc_tmp >= 0.70:
        best_tn, best_thr, best_acc = tn_rate, thr, acc_tmp
print(f"Optimal threshold: {best_thr:.3f}  (TN rate={best_tn:.3f}, acc={best_acc:.3f})")

y_pred = (proba_test >= best_thr).astype(int)


In [ ]:
# Cross-validation
cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
for metric in ['accuracy','precision','recall','f1']:
    scores = cross_val_score(rf, Xtr_s, y_train, cv=cv, scoring=metric, n_jobs=-1)
    print(f"  CV {metric:10s}: {scores.mean():.4f} ± {scores.std():.4f}")


In [ ]:
# Final evaluation
acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)
auc  = roc_auc_score(y_test, proba_test)
cm   = confusion_matrix(y_test, y_pred)

print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1:        {f1:.4f}")
print(f"AUC-ROC:   {auc:.4f}")
print()
print(classification_report(y_test, y_pred,
      target_names=['Unbarred','Barred'], digits=4, zero_division=0))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Unbarred','Barred'],
            yticklabels=['Unbarred','Barred'], ax=axes[0])
axes[0].set_title(f'Confusion Matrix (acc={acc:.3f})')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, proba_test)
axes[1].plot(fpr, tpr, lw=2, label=f'AUC={auc:.3f}')
axes[1].plot([0,1],[0,1],'k--', lw=1)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend()
axes[1].grid(alpha=0.3)

imp = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
imp.head(12).plot.barh(ax=axes[2], color='steelblue')
axes[2].invert_yaxis()
axes[2].set_title('Top 12 Feature Importances')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.savefig('rf_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("Plot saved.")


In [ ]:
# Save outputs
pd.DataFrame({
    'Target_Name':      test_names,
    'True_Label':       y_test,
    'Predicted_Label':  y_pred,
    'Probability_Barred': proba_test,
    'Correct':          y_test == y_pred,
}).to_csv('rf_predictions.csv', index=False)

imp.reset_index().rename(columns={'index':'Feature',0:'Importance'}).to_csv(
    'rf_feature_importance.csv', index=False)

print("Saved: rf_predictions.csv, rf_feature_importance.csv, rf_results.png")
